# Lesson 10 - AI Agents for Production

For dis lesson, you go learn **production patterns** for AI agents using Microsoft Agent Framework (Python). We go cover:

- **Observability** — to add timing and logging for agent interactions
- **Evaluation** — to use evaluator agent to score how beta response be
- **Cost management** — how to manage token plus model selection

The scenario na **travel agent** wey dey help users plan trip, with monitoring plus evaluation join for top.


## Setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Production Considerations

Moving AI agents from prototypes to production require careful attention to three pillars:

1. **Observability** — You need visibility into wetin the agent dey do, how long e take, and which tools e dey call. Without tracing and logging, debugging production issues hard well well.

2. **Evaluation** — Automated quality checks dey make sure say the agent's responses dey accurate, complete, and helpful over time. One evaluator agent fit score responses against defined criteria.

3. **Cost Management** — Token use dey affect cost directly. Strategies like prompt optimization, model selection, and caching dey help keep expenses under control without losing quality.


## Building an Observable Agent

We dey define travel tools and wrap di agent call wit timing so we fit monitor latency. For production, you go integrate wit OpenTelemetry or similar tracing backend.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evaluation Patterns

One common way wey people dey use for production na to get another agent as **evaluator**. The evaluator dey score how the primary agent respond based on criteria like if e complete, correct, and if e help.

Dis one fit make:
- Automated quality checks before responses reach users
- Detect regression when prompts or models change
- Continuous monitoring of how well the agent dey perform over time


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Cost Management Strategies

Control cost na serious mata for production AI agents. Here be the main strategies:

| Strategy | Description |
|---|---|
| **Prompt optimization** | Make system instructions short and clean. Comot unnecessary context to reduce input tokens. |
| **Model selection** | Use smaller, cheap models (like GPT-4o-mini) for simple work like classification or extraction, and keep bigger models for heavy reasoning. |
| **Caching** | Save tool results and frequent questions to avoid calling API again and again. |
| **Token budgets** | Set `max_tokens` limit make e no too long for unexpected long answers. |
| **Batching** | Group plenty user questions into one API call if e possible. |

For practice, the best way na to arrange am by level: send simple requests to fast, cheap model and only move complex questions go better model.


## Summary

For dis lesson, you learn how to:

1. **Add observability** for agent interactions wit timing and logging, wey go lay di foundation for tracing and monitoring.
2. **Evaluate agent responses** automatically using evaluator agent wey dey score completeness, accuracy, and helpfulness.
3. **Manage costs** through prompt optimization, model selection, caching, and token budgets.

These production patterns dey help make sure say your AI agents dey reliable, measurable, and cost-effective for big scale.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
